In [98]:
import numpy as np
import pandas as pd
import neurokit2 as nk

Imported reshape function as it will be needed to make row-wise descriptives

In [99]:
#Now we make the reshape function
#4 parameters, your dataframe, your window size or how many hz of rows should be in each row, purity for how much each row can tolerate label confounsion, and stride size for how much we should overlap
#By default this is every 10 seconds of data
def reshape_function(df, name = 'eda', window_size = 60, stride_size = 15, hertz=8, purity_threshold=0.8):
    window_row = window_size*hertz
    stride_row = stride_size*hertz
    #Get the subject names
    subjects = sorted(
        list(set(col.split("_")[0] for col in df.columns if "_" in col))
    )
    #Store the chunks of data while we're looping across the time span
    all_window = []
    for i in subjects:
        signal_col = f'{i}_{name}'
        label_col = f'{i}_Label'
        #Kill NAs in individual sequence
        individual_df = df[[signal_col,label_col]].dropna()
        total_rows = len(individual_df)
        #For each subject we take a chunk of their data by our determined window size then stack onto a list (in this case "all window")
        #Chunk is determined through standard range check then we check the purity threshold if we keep the chunk, based on birken's paper design
        for j in range(0,total_rows-window_row+1, stride_row):
            end = j + window_row
            signal_window = individual_df[signal_col].iloc[j:end].values
            label_window = individual_df[label_col].iloc[j:end].values

            # Determine the majority label and calculate if there exists a supermajority in the chunk to determine if kept
            value_counts = pd.Series(label_window).value_counts()
            majority_label = value_counts.index[0]
            majority_count = value_counts.iloc[0]

            purity = majority_count / window_row

            # Retain only windows meeting or exceeding the requested purity ratio
            if purity >= purity_threshold:
                row_data = [i, majority_label] + list(signal_window)
                all_window.append(row_data)

    feature_cols = [f"sample_{i}" for i in range(window_row)]
    columns = ["subject_id", "label"] + feature_cols

    return pd.DataFrame(all_window, columns=columns)

    





I also want visualization boxplots for each feature set similar to Zhu lili's breakdown

Import tonic and phasic files

In [100]:
tonic = pd.read_csv("../Data/wesad_eda/tonic_signal.csv")
phasic = pd.read_csv("../Data/wesad_eda/phasic_signal.csv")

This section is just applying the reshape function and verifying the outputs

In [101]:
tonic_10 = reshape_function(tonic, name='Tonic')

In [102]:
tonic_10.shape

(2106, 482)

In [103]:
tonic_10.head()

,subject_id,label,sample_0,sample_1,sample_2,sample_3,sample_4,sample_5,sample_6,sample_7,...,sample_470,sample_471,sample_472,sample_473,sample_474,sample_475,sample_476,sample_477,sample_478,sample_479
0,S10,1.0,0.306431,0.305287,0.304148,0.303015,0.301888,0.300767,0.299654,0.298550,...,0.378733,0.378793,0.378852,0.378911,0.378971,0.379029,0.379088,0.379145,0.379203,0.379259
1,S10,1.0,0.310498,0.311419,0.312340,0.313260,0.314179,0.315097,0.316015,0.316930,...,0.375171,0.375122,0.375076,0.375033,0.374992,0.374955,0.374920,0.374889,0.374861,0.374836
2,S10,1.0,0.377523,0.377557,0.377585,0.377608,0.377626,0.377639,0.377648,0.377653,...,0.377676,0.377650,0.377624,0.377597,0.377570,0.377544,0.377517,0.377490,0.377464,0.377437
3,S10,1.0,0.377331,0.377335,0.377338,0.377340,0.377340,0.377340,0.377339,0.377336,...,0.386250,0.386416,0.386583,0.386750,0.386917,0.387083,0.387250,0.387417,0.387583,0.387749
4,S10,1.0,0.379315,0.379371,0.379425,0.379479,0.379532,0.379583,0.379634,0.379684,...,0.392702,0.392570,0.392437,0.392302,0.392166,0.392028,0.391890,0.391750,0.391609,0.391468


Tonic descriptives row-wise for max and mean

In [104]:
sample_cols = [col for col in tonic_10.columns if col.startswith('sample_')]
tonic_descriptives = pd.DataFrame()
#Bring Labels into descriptives to the databas
tonic_descriptives['label'] = tonic_10['label']
#Bring subject ID over to descriptives
if 'subject_id' in tonic_10.columns:
    tonic_descriptives['subject_id'] = tonic_10['subject_id']
#Descriptives
tonic_descriptives['tonic_mean'] = tonic_10[sample_cols].mean(axis=1)
tonic_descriptives['tonic_max']  = tonic_10[sample_cols].max(axis=1)

tonic_descriptives.head()

,label,subject_id,tonic_mean,tonic_max
0,1.0,S10,0.347957,0.379259
1,1.0,S10,0.371636,0.380368
2,1.0,S10,0.377360,0.380368
3,1.0,S10,0.378196,0.387749
4,1.0,S10,0.382147,0.395566


In [105]:
phasic_10 = reshape_function(phasic, name='Phasic')

Phasic descriptives row-wise for mean, peak counts and rise time standard deviation

In [106]:
phasic_10.shape

(2106, 482)

In [107]:
sample_cols = [col for col in phasic_10.columns if col.startswith('sample_')]
phasic_descriptives = pd.DataFrame()
#Bring Labels into descriptives to the database
phasic_descriptives['label'] = phasic_10['label']
#Bring subject ID over to descriptives
if 'subject_id' in phasic_10.columns:
    phasic_descriptives['subject_id'] = phasic_10['subject_id']
#Descriptives
phasic_descriptives['phasic_mean'] = phasic_10[sample_cols].mean(axis=1)
#phasic_descriptives['peaks_count'] = (phasic_10[sample_cols] > 0.01).sum(axis=1)
phasic_descriptives.head()

,label,subject_id,phasic_mean
0,1.0,S10,0.029461
1,1.0,S10,0.003448
2,1.0,S10,0.000585
3,1.0,S10,0.001437
4,1.0,S10,0.001244


Test example before I implemented on full dataframe the rise time standard deviation descriptive

In [108]:
row_example = phasic_10.iloc[0:1,2:]
signals, info = nk.eda_peaks(np.array(row_example).flatten(), sampling_rate=4)
# Calculate rise times in seconds
rise_times_sec = (info["SCR_Peaks"] - info["SCR_Onsets"]) / 4
np.nanstd(rise_times_sec)


1.143368561167687

In [109]:
def get_eda_peak_metrics(row, sampling_rate=8):
    try:
        # Detect peaks and onsets using NeuroKit2
        signals, info = nk.eda_peaks(row, sampling_rate=sampling_rate)
        
        # Count the number of peaks detected in the row
        num_peaks = len(info["SCR_Peaks"])
        
        if num_peaks > 0:
            # Calculate rise times in seconds
            rise_times = (info["SCR_Peaks"] - info["SCR_Onsets"]) / sampling_rate
            std_rise_time = np.nanstd(rise_times)
        else:
            std_rise_time = np.nan

        return pd.Series([num_peaks, std_rise_time], index=['peak_count', 'std_rise_time'])
        
    except Exception:
        # Return NaN for both metrics if processing fails
        return pd.Series([np.nan, np.nan], index=['peak_count', 'std_rise_time'])

# Apply the function across rows and assign to new columns
phasic_descriptives[['phasic_peak_count', 'phasic_std_rise_time']] = phasic_10[sample_cols].apply(
    lambda row: get_eda_peak_metrics(row), axis=1
)

phasic_descriptives.head()

/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
/home/sj/.local/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
/home/sj/.local/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice

,label,subject_id,phasic_mean,phasic_peak_count,phasic_std_rise_time
0,1.0,S10,0.029461,7.0,0.571684
1,1.0,S10,0.003448,9.0,0.688742
2,1.0,S10,0.000585,2.0,0.187500
3,1.0,S10,0.001437,5.0,1.728665
4,1.0,S10,0.001244,5.0,1.570430


Create and save file descriptive file

In [110]:
feature_dataframe = pd.concat(
    [phasic_descriptives, tonic_descriptives.drop(columns=["subject_id", "label"])],
    axis=1,
)
feature_dataframe.head()

,label,subject_id,phasic_mean,phasic_peak_count,phasic_std_rise_time,tonic_mean,tonic_max
0,1.0,S10,0.029461,7.0,0.571684,0.347957,0.379259
1,1.0,S10,0.003448,9.0,0.688742,0.371636,0.380368
2,1.0,S10,0.000585,2.0,0.187500,0.377360,0.380368
3,1.0,S10,0.001437,5.0,1.728665,0.378196,0.387749
4,1.0,S10,0.001244,5.0,1.570430,0.382147,0.395566


In [111]:
feature_dataframe.to_csv("../Data/wesad_eda/feature_dataframe.csv", index=False)
